In [ ]:
"""Off-target activation of the cardiometabolic experts"""
import os
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

from help.ckm_utils import CONDITIONS
from help.stats_utils import SEED, N_BOOT
from help.system_defs import add_system_flags

In [ ]:
MIN_GROUP_N, UNDERPOWERED_N = 5, 50
# MIN_GROUP_N: minimum n in both the condition group and the clean-control group to attempt
# a comparison (below this, the comparison is skipped entirely, not just flagged)
# UNDERPOWERED_N: higher floor below which Cohen's d IS reported but marked "underpowered"
# in the output tables, and excluded from the BH-FDR family

POP_PATH, OOF_PATH = "analytic_population.pkl", "Results_MoE/oof_predictions.csv"   
OUT_DIR = "Results_Specificity_BySystem"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ---- 1. Load ----
calib_cols = [f"calib_{c}" for c in CONDITIONS]
pop = add_system_flags(pd.read_pickle(POP_PATH))
pop["gender"] = pop["gender"].str.lower().str.strip()
oof = pd.read_csv(OOF_PATH)[["participant_id", "burden_score", *calib_cols]]
df = pop.drop(columns="burden_score", errors="ignore").merge(oof, on="participant_id", how="inner", validate="1:1")
assert len(df) == len(pop) == len(oof), "participants differ between the population file and 02_train output"
assert df["age"].notna().all()
df["z_Aggregate"] = df[calib_cols].mean(axis=1)
df["has_cvd"] = (df["cvdd_count"] > 0).astype(float).where(df["cvdd_count"].notna())
Z_COLS = {**{c: f"calib_{c}" for c in CONDITIONS}, "Aggregate": "z_Aggregate"}

In [ ]:
# ---- 2. Groups (system level only) ----
def build_groups(d):
    b0 = d["burden_score"] == 0
    mental, resp = d["has_any_mental"] == 1, d["has_any_respiratory"] == 1
    clean = b0 & ~mental & ~resp
    groups = {"Mental": (b0 & mental & ~resp, clean),
              "Respiratory": (b0 & resp & ~mental, clean),
              "CVD (positive control)": ((d["has_cvd"] == 1) & ~mental & ~resp, clean)}
    return groups

In [ ]:
# ---- 3. Effect sizes, bootstrap (with bootstrap p for FDR) ----
def cohens_d(g, b):
    return (g.mean() - b.mean()) / np.sqrt((g.var(ddof=1) + b.var(ddof=1)) / 2)

def compare(g, b, stat=cohens_d, n_boot=N_BOOT, seed=SEED):
    g, b = g[np.isfinite(g)], b[np.isfinite(b)]
    rng = np.random.default_rng(seed)
    reps = np.array([stat(g[rng.integers(0, len(g), len(g))], b[rng.integers(0, len(b), len(b))]) for _ in range(n_boot)])
    reps = reps[np.isfinite(reps)]
    lo, hi = np.percentile(reps, [2.5, 97.5])
    p_boot = max(2 * min((reps <= 0).mean(), (reps >= 0).mean()), 1 / (len(reps) + 1))   # two-sided bootstrap p (floor 1/(B+1))
    return dict(n_group=len(g), n_clean=len(b), estimate=stat(g, b), ci_lo=lo, ci_hi=hi, p_boot=p_boot,
                ci_excludes_0=bool(lo > 0 or hi < 0), underpowered=min(len(g), len(b)) < UNDERPOWERED_N)

In [ ]:
# ---- 4. Specificity table ----
dfs = {sex: d.reset_index(drop=True) for sex, d in df.groupby("gender")}
groups_by_sex = {sex: build_groups(d) for sex, d in dfs.items()}

rows = []
for sex, d in dfs.items():
    for group, (gm, cm) in groups_by_sex[sex].items():
        gmv, cmv = gm.to_numpy(), cm.to_numpy()
        if gmv.sum() < MIN_GROUP_N or cmv.sum() < MIN_GROUP_N:
            continue
        for expert, zcol in Z_COLS.items():
            z = d[zcol].to_numpy(float)
            rows.append({"sex": sex, "group": group, "expert": expert, **compare(z[gmv], z[cmv])})
specificity = pd.DataFrame(rows)

# BH-FDR on the powered family (CVD control excluded)
fam = (~specificity.underpowered) & (~specificity.group.str.startswith("CVD"))
specificity["q_bh"] = np.nan
specificity.loc[fam, "q_bh"] = multipletests(specificity.loc[fam, "p_boot"], method="fdr_bh")[1]
specificity["robust"] = specificity.ci_excludes_0 & (specificity.q_bh < 0.05)
specificity.to_csv(f"{OUT_DIR}/specificity_by_group.csv", index=False)

valid = specificity[~specificity.underpowered & ~specificity.group.str.startswith("CVD")]

flag_tbl = specificity[specificity.ci_excludes_0].copy()
flag_tbl["d (95% CI)"] = flag_tbl.apply(lambda r: f"{r.estimate:+.2f} ({r.ci_lo:+.2f} to {r.ci_hi:+.2f})", axis=1)
flag_tbl["note"] = np.where(flag_tbl.underpowered, "underpowered", np.where(flag_tbl.robust, "robust (q<0.05)", "CI only"))
flag_tbl.to_csv(f"{OUT_DIR}/specificity_flagged.csv", index=False)

In [ ]:
# ---- 5. Summary ----
# Aggregate d by group/sex
order = [("Mental", "female"), ("Respiratory", "female"), ("CVD (positive control)", "female"),
         ("Mental", "male"), ("Respiratory", "male"), ("CVD (positive control)", "male")]
agg = specificity[specificity.expert == "Aggregate"].set_index(["group", "sex"])
agg_tbl = pd.DataFrame([
    {"group": g, "sex": s, "n_group": int(agg.loc[(g, s), "n_group"]),
     "d (95% CI)": f"{agg.loc[(g, s), 'estimate']:+.2f} ({agg.loc[(g, s), 'ci_lo']:+.2f} to {agg.loc[(g, s), 'ci_hi']:+.2f})"}
    for g, s in order if (g, s) in agg.index
])
print("\nAggregate z-score vs burden-zero controls, by group and sex:")
print(agg_tbl.to_string(index=False))

cvd_ok = specificity[specificity.group.str.startswith("CVD")]
print(f"\nCVD positive control: {int(cvd_ok.ci_excludes_0.sum())} of {len(cvd_ok)} expert/aggregate "
      f"comparisons show the expected off-target rise (underpowered, descriptive only, n<50 per arm).")